# GPU timing: NeuralHydrology and Pangu-Weather (Colab T4)

The same benchmarks as on the laptop and the CSE server, so all three machines can be compared:
* **NeuralHydrology**: basin 3002 alone (20 hidden units) and 10 basins (128 hidden units), real CAMELS-IND data.
* **Pangu-Weather**: time per 6-hour forecast step on the GPU (the number missing from the report).

**Runtime → Change runtime type → T4 GPU**, then run all cells. About 20–30 minutes.

In [ ]:
# 1. GPU, Drive and packages
import torch, os
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch to a T4 GPU runtime!")
from google.colab import drive
drive.mount('/content/drive')
!pip -q install neuralhydrology
!pip -q uninstall -y onnxruntime
!pip -q install onnxruntime-gpu
import onnxruntime as ort; print("onnxruntime", ort.__version__, ort.get_available_providers())
OUT = '/content/drive/MyDrive/ugp_timing'
os.makedirs(OUT, exist_ok=True)

In [ ]:
%%writefile /content/nh_multibasin.py
"""Multi-basin LSTM on CAMELS-IND with NeuralHydrology (helpers used by the Colab notebook).

Steps
  1. prepare()        raw CAMELS-IND -> NeuralHydrology layout -> streamflow quality control ->
                      basin selection -> input / attribute selection        (data_dir: <work>/camels_ind_qc)
  2. write_configs()  YAML files for the multi-basin model and the single-basin comparison models
  3. analyse()        test scores per basin, single vs multi comparison, figures and tables

Rain input: CAMELS-IND's `prcp` is the IMD 0.25 deg gauge-based gridded rainfall averaged over each
catchment, so the model is driven by IMD rainfall averaged over the catchment (not one grid cell).
"""
import glob
import json
import os
import pickle
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

TARGET = "streamflow_mm_day"
PERIODS = dict(train=("01/01/1981", "31/12/2004"),
               validation=("01/01/2005", "31/12/2009"),
               test=("01/01/2010", "31/12/2020"))

# ---------------------------------------------------------------- static attributes
# Concepts to look for (first matching column name wins). Streamflow signatures are never used,
# because they are computed from the very flow the model has to predict.
ATTR_CONCEPTS = [
    ["cwc_area", "area"], ["elev_mean", "elev"], ["slope_mean", "slope"], ["dpsbar", "drainage"],
    ["p_mean", "prec_mean", "precip_mean"], ["pet_mean"], ["aridity"], ["p_seasonality", "seasonality"],
    ["asynchronicity", "asynch"], ["high_prec_freq"], ["high_prec_dur"], ["low_prec_freq", "dry_day"],
    ["low_prec_dur"], ["frac_forest", "forest"], ["crop", "agri"], ["urban", "built"], ["water_frac", "water"],
    ["lai_max", "lai"], ["sand"], ["silt"], ["clay"], ["soil_depth", "depth"], ["awc", "water_storage", "available"],
    ["conductivity", "ksat"], ["porosity"], ["permeab"], ["dam_count", "num_dams", "dams"],
    ["storage", "reservoir"], ["pop", "population"],
]
SIGNATURE_WORDS = ["q_mean", "runoff", "baseflow", "bfi", "flow", "elasticity", "fdc", "gini", "hfd",
                   "q5", "q95", "zero_q", "high_q", "low_q", "streamflow", "discharge"]
SKIP_WORDS = ["gauge", "lat", "lon", "id", "name", "code", "state", "river", "basin_name"]


def _find_base(raw_dir: Path) -> Path:
    for p in [raw_dir] + sorted(raw_dir.rglob("*")):
        if p.is_dir() and (p / "catchment_mean_forcings").is_dir() and (p / "attributes_csv").is_dir():
            return p
    raise FileNotFoundError(f"No folder with catchment_mean_forcings/ and attributes_csv/ inside {raw_dir}")


def _read_attribute_tables(base: Path) -> tuple[pd.DataFrame, dict]:
    tables, origin = [], {}
    for f in sorted((base / "attributes_csv").glob("*.csv")):
        df = pd.read_csv(f)
        if "gauge_id" not in df.columns:
            continue
        df["gauge_id"] = df["gauge_id"].astype(str)
        df = df.drop_duplicates("gauge_id").set_index("gauge_id")
        for c in df.columns:
            origin.setdefault(c, f.stem)
        tables.append(df)
    attrs = pd.concat(tables, axis=1)
    attrs = attrs.loc[:, ~attrs.columns.duplicated()]
    return attrs, origin


def _pick_attributes(attrs: pd.DataFrame, origin: dict, basins: list, max_nan=0.10) -> list:
    sub = attrs.loc[basins]
    numeric = [c for c in sub.columns if pd.api.types.is_numeric_dtype(sub[c])]

    def usable(c):
        lc, lo = c.lower(), origin.get(c, "").lower()
        if any(w in lc for w in SIGNATURE_WORDS) or "hydro" in lo or "signature" in lo:
            return False
        if any(lc == w or lc.startswith(w + "_") or lc.endswith("_" + w) for w in SKIP_WORDS):
            return False
        col = sub[c]
        return col.isna().mean() <= max_nan and col.std(skipna=True) > 0

    chosen = []
    for concept in ATTR_CONCEPTS:
        for key in concept:
            hits = [c for c in numeric if key in c.lower() and usable(c) and c not in chosen]
            if hits:
                chosen.append(sorted(hits, key=len)[0])
                break
    if len(chosen) < 8:                      # unknown naming: fall back to the most complete columns
        extra = [c for c in numeric if usable(c) and c not in chosen]
        extra = sorted(extra, key=lambda c: sub[c].isna().mean())[: 25 - len(chosen)]
        chosen += extra
    return chosen


# ---------------------------------------------------------------- quality control
def qc_streamflow(df: pd.DataFrame, rain_col: str):
    """Returns cleaned flow and a dict of counts. Flags:
       negative values; identical extreme values repeated on separate days (the 946.46722 mm/day case);
       daily flow more than 100 mm above the rain of the previous 7 days (impossible without a data error)."""
    q = df[TARGET].astype(float).copy()
    n_obs = int(q.notna().sum())
    flags = pd.Series(False, index=q.index)
    neg = q < 0
    flags |= neg
    rep = pd.Series(False, index=q.index)
    if n_obs > 100:
        thr = q.quantile(0.999)
        big = q[(q >= thr) & (q > 20)]
        counts = big.round(5).value_counts()
        rep_vals = counts[counts >= 2].index
        rep = q.round(5).isin(rep_vals) & (q > 20)
        flags |= rep
    rain7 = df[rain_col].rolling(7, min_periods=1).sum() if rain_col in df else None
    over = (q > rain7 + 100) if rain7 is not None else pd.Series(False, index=q.index)
    flags |= over.fillna(False)
    q[flags] = np.nan
    return q, dict(n_obs=n_obs, negative=int(neg.sum()), repeated_extreme=int(rep.sum()),
                   flow_above_rain=int(over.fillna(False).sum()), removed=int(flags.sum()))


def _count_valid(q: pd.Series, period: str) -> int:
    a, b = (pd.to_datetime(x, dayfirst=True) for x in PERIODS[period])
    return int(q.loc[a:b].notna().sum())


def prepare(raw_dir: str, work_dir: str, min_train_days=3 * 365, min_test_days=365, max_runoff_ratio=1.0):
    from neuralhydrology.datasetzoo.camelsind import preprocess_camels_ind_dataset
    raw_dir, work = Path(raw_dir), Path(work_dir)
    base = _find_base(raw_dir)
    proc = work / "camels_ind_proc"
    if not (proc / "preprocessed").is_dir():
        if proc.exists():
            shutil.rmtree(proc)
        preprocess_camels_ind_dataset(str(base), str(proc))
    qc_dir = work / "camels_ind_qc"
    (qc_dir / "preprocessed").mkdir(parents=True, exist_ok=True)

    files = sorted((proc / "preprocessed").glob("*.csv"))
    sample = pd.read_csv(files[0], nrows=5)
    rain_col = next((c for c in sample.columns if c.lower() in ("prcp", "precipitation", "prec", "rain")),
                    next((c for c in sample.columns if c.lower().startswith("prcp")), None))
    print(f"{len(files)} basin files; rain column: {rain_col}")

    report, data = [], {}
    for f in files:
        df = pd.read_csv(f, index_col="date", parse_dates=["date"])
        if TARGET not in df or df[TARGET].notna().sum() == 0:
            continue
        q, counts = qc_streamflow(df, rain_col)
        df[TARGET] = q
        both = df[[TARGET, rain_col]].dropna()
        rr = both[TARGET].mean() / both[rain_col].mean() if len(both) > 365 and both[rain_col].mean() > 0 else np.nan
        row = dict(basin=f.stem, **counts, runoff_ratio=round(float(rr), 3) if rr == rr else np.nan,
                   train_days=_count_valid(q, "train"), val_days=_count_valid(q, "validation"),
                   test_days=_count_valid(q, "test"))
        reason = []
        if row["train_days"] < min_train_days:
            reason.append("short train record")
        if row["test_days"] < min_test_days:
            reason.append("short test record")
        if not (rr <= max_runoff_ratio):
            reason.append("runoff ratio > 1 or unknown")
        row["kept"] = not reason
        row["reason"] = "; ".join(reason)
        report.append(row)
        data[f.stem] = df

    rep = pd.DataFrame(report)
    rep.to_csv(work / "qc_report.csv", index=False)
    basins = rep.loc[rep.kept, "basin"].tolist()
    print(f"Quality control: {int(rep.removed.sum())} flow values removed in {int((rep.removed > 0).sum())} basins "
          f"(negative {int(rep.negative.sum())}, repeated extremes {int(rep.repeated_extreme.sum())}, "
          f"flow > 7-day rain + 100 mm {int(rep.flow_above_rain.sum())})")
    print(f"Basins with flow: {len(rep)}; kept: {len(basins)}; dropped: "
          + ", ".join(f"{k} ({v})" for k, v in rep.loc[~rep.kept, "reason"].value_counts().items()))
    if "3002" in data:
        r = rep.set_index("basin").loc["3002"]
        print(f"Basin 3002: removed {r.removed} values (repeated extremes {r.repeated_extreme}); kept={r.kept}")

    # dynamic inputs: every forcing column that is complete for the kept basins
    cols = [c for c in data[basins[0]].columns if c != TARGET]
    a, b = pd.Timestamp("1980-01-01"), pd.Timestamp("2020-12-31")
    nan_frac = {c: np.mean([data[k].loc[a:b, c].isna().mean() for k in basins]) for c in cols}
    dyn = [c for c in cols if nan_frac[c] < 0.01]
    print(f"Dynamic inputs ({len(dyn)}): {dyn}")
    if len(dyn) < len(cols):
        print(f"  left out (incomplete): {[c for c in cols if c not in dyn]}")
    for k in basins:
        data[k][dyn + [TARGET]].to_csv(qc_dir / "preprocessed" / f"{k}.csv")

    attrs, origin = _read_attribute_tables(base)
    stat = _pick_attributes(attrs, origin, basins)
    out = attrs.loc[basins, stat].copy()
    out = out.fillna(out.median(numeric_only=True))
    out.index.name = "gauge_id"
    out.to_csv(qc_dir / "attributes.csv")
    print(f"Static attributes ({len(stat)}): {stat}")

    (work / "basins_all.txt").write_text("\n".join(basins) + "\n")
    info = dict(basins=basins, dynamic_inputs=dyn, static_attributes=stat, rain_col=rain_col,
                data_dir=str(qc_dir))
    (work / "prep_info.json").write_text(json.dumps(info, indent=1))
    return info


# ---------------------------------------------------------------- configs
def _yaml_list(x):
    return "\n".join(f"  - {v}" for v in x)


def _config(name, run_dir, basin_file, data_dir, dyn, stat, hidden, loss, epochs, lr, device, seed,
            num_workers, validate_n=None):
    s = f"""experiment_name: {name}
run_dir: {run_dir}
train_basin_file: {basin_file}
validation_basin_file: {basin_file}
test_basin_file: {basin_file}
train_start_date: '{PERIODS['train'][0]}'
train_end_date: '{PERIODS['train'][1]}'
validation_start_date: '{PERIODS['validation'][0]}'
validation_end_date: '{PERIODS['validation'][1]}'
test_start_date: '{PERIODS['test'][0]}'
test_end_date: '{PERIODS['test'][1]}'
seed: {seed}
device: {device}
validate_every: 5
{"validate_n_random_basins: " + str(validate_n) if validate_n else ""}
metrics:
  - NSE
  - KGE
model: cudalstm
head: regression
output_activation: linear
hidden_size: {hidden}
initial_forget_bias: 3
output_dropout: 0.4
optimizer: Adam
loss: {loss}
learning_rate:
{chr(10).join(f'  {k}: {v}' for k, v in lr.items())}
batch_size: 256
epochs: {epochs}
clip_gradient_norm: 1
predict_last_n: 1
seq_length: 365
num_workers: {num_workers}
log_interval: 5
log_tensorboard: False
log_n_figures: 0
save_weights_every: {epochs}
dataset: camels_ind
data_dir: {data_dir}
dynamic_inputs:
{_yaml_list(dyn)}
target_variables:
  - {TARGET}
clip_targets_to_zero:
  - {TARGET}
"""
    if stat:
        s += "static_attributes:\n" + _yaml_list(stat) + "\n"
    return s


def write_configs(work_dir, run_dir, device="cuda:0", n_single=10, single_extra=("3002",), seeds=(1,),
                  multi_hidden=128, multi_epochs=30, single_epochs=50, num_workers=2, rng_seed=0):
    work = Path(work_dir); info = json.loads((work / "prep_info.json").read_text())
    cfg_dir = work / "configs"; cfg_dir.mkdir(exist_ok=True)
    Path(run_dir).mkdir(parents=True, exist_ok=True)
    paths = []
    for sd in seeds:
        p = cfg_dir / f"multi_seed{sd}.yml"
        p.write_text(_config(f"multi_seed{sd}", run_dir, work / "basins_all.txt", info["data_dir"],
                             info["dynamic_inputs"], info["static_attributes"], multi_hidden, "NSE",
                             multi_epochs, {0: 1e-3, int(multi_epochs * 0.6): 5e-4, int(multi_epochs * 0.85): 1e-4},
                             device, sd, num_workers, validate_n=min(50, len(info["basins"]))))
        paths.append(p)
    rnd = random.Random(rng_seed)
    singles = [b for b in single_extra if b in info["basins"]]
    singles += rnd.sample([b for b in info["basins"] if b not in singles], min(n_single, len(info["basins"]) - len(singles)))
    for b in singles:
        bf = cfg_dir / f"basin_{b}.txt"; bf.write_text(b + "\n")
        p = cfg_dir / f"single_{b}.yml"
        p.write_text(_config(f"single_{b}", run_dir, bf, info["data_dir"], info["dynamic_inputs"], [], 20, "MSE",
                             single_epochs, {0: 1e-2, int(single_epochs * 0.6): 5e-3, int(single_epochs * 0.8): 1e-3},
                             device, 1, num_workers))
        paths.append(p)
    (work / "single_basins.txt").write_text("\n".join(singles) + "\n")
    print(f"Wrote {len(paths)} configs in {cfg_dir}: multi-basin x{len(seeds)}, single-basin x{len(singles)} "
          f"({', '.join(singles)})")
    return paths


# ---------------------------------------------------------------- analysis
def _metrics(obs, sim):
    m = ~np.isnan(obs) & ~np.isnan(sim)
    o, s = obs[m], sim[m]
    if len(o) < 30 or np.var(o) == 0:
        return dict(n=int(len(o)), NSE=np.nan, KGE=np.nan, r=np.nan)
    nse = 1 - np.sum((s - o) ** 2) / np.sum((o - o.mean()) ** 2)
    r = np.corrcoef(o, s)[0, 1] if np.std(s) > 0 else 0.0
    alpha = np.std(s) / np.std(o); beta = s.mean() / o.mean() if o.mean() != 0 else np.nan
    kge = 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)
    return dict(n=int(len(o)), NSE=nse, KGE=kge, r=r)


def _latest_run(run_dir, name):
    runs = sorted(glob.glob(os.path.join(run_dir, f"{name}_*")))
    return Path(runs[-1]) if runs else None


def _load_test(run):
    files = sorted(run.glob("test/model_epoch*/test_results.p"))
    if not files:
        return None
    with open(files[-1], "rb") as fh:
        res = pickle.load(fh)
    out = {}
    for b, d in res.items():
        xr = d["1D"]["xr"]
        obs = xr[f"{TARGET}_obs"].values[:, -1]; sim = xr[f"{TARGET}_sim"].values[:, -1]
        out[b] = pd.DataFrame({"obs": obs, "sim": sim}, index=pd.to_datetime(xr["date"].values))
    return out


def analyse(work_dir, run_dir, out_dir, window_3002=("2017-01-01", "2018-10-08")):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    work, out = Path(work_dir), Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    singles = (work / "single_basins.txt").read_text().split()

    multi_runs = sorted({Path(p).name.rsplit("_", 2)[0] for p in glob.glob(os.path.join(run_dir, "multi_seed*"))})
    multi = [_load_test(_latest_run(run_dir, n)) for n in multi_runs]
    multi = [m for m in multi if m]
    if not multi:
        raise RuntimeError("No evaluated multi-basin run found (run nh-run evaluate first).")
    basins = sorted(multi[0])
    ens = {b: pd.DataFrame({"obs": multi[0][b]["obs"],
                            "sim": np.mean([m[b]["sim"].values for m in multi], axis=0)}, index=multi[0][b].index)
           for b in basins}
    rows = [dict(basin=b, model=f"multi-basin ({len(multi)} seed{'s' if len(multi) > 1 else ''})",
                 **_metrics(ens[b]["obs"].values, ens[b]["sim"].values)) for b in basins]
    single = {}
    for b in singles:
        run = _latest_run(run_dir, f"single_{b}")
        res = _load_test(run) if run else None
        if res and b in res:
            single[b] = res[b]
            rows.append(dict(basin=b, model="single-basin", **_metrics(res[b]["obs"].values, res[b]["sim"].values)))
    tab = pd.DataFrame(rows)
    tab.to_csv(out / "test_scores_per_basin.csv", index=False)

    mname = tab.model.iloc[0]
    m_all = tab[tab.model == mname]
    m_sub = m_all[m_all.basin.isin(single)]
    s_sub = tab[tab.model == "single-basin"]
    summ = pd.DataFrame([
        dict(group=f"Multi-basin, all {len(m_all)} basins", **{f"median_{k}": m_all[k].median() for k in ["NSE", "KGE", "r"]},
             share_NSE_gt_0=(m_all.NSE > 0).mean(), share_NSE_gt_05=(m_all.NSE > 0.5).mean()),
        dict(group=f"Multi-basin, the {len(m_sub)} comparison basins", **{f"median_{k}": m_sub[k].median() for k in ["NSE", "KGE", "r"]},
             share_NSE_gt_0=(m_sub.NSE > 0).mean(), share_NSE_gt_05=(m_sub.NSE > 0.5).mean()),
        dict(group=f"Single-basin, the same {len(s_sub)} basins", **{f"median_{k}": s_sub[k].median() for k in ["NSE", "KGE", "r"]},
             share_NSE_gt_0=(s_sub.NSE > 0).mean(), share_NSE_gt_05=(s_sub.NSE > 0.5).mean()),
    ])
    summ.to_csv(out / "summary.csv", index=False)
    pd.set_option("display.width", 160)
    print(f"\nTest period {PERIODS['test'][0]} - {PERIODS['test'][1]}\n")
    print(summ.round(3).to_string(index=False))

    pair = m_sub.merge(s_sub, on="basin", suffixes=("_multi", "_single"))
    if len(pair):
        print(f"\nMulti-basin better than single-basin (NSE) in {(pair.NSE_multi > pair.NSE_single).sum()} "
              f"of {len(pair)} basins")
        print(pair[["basin", "NSE_single", "NSE_multi", "KGE_single", "KGE_multi"]].round(3).to_string(index=False))

    if "3002" in ens:
        a, b = window_3002
        print(f"\nBasin 3002, {a} to {b} (same window as the FNL / CAMELS-IND single-basin study):")
        lines = [("multi-basin", ens["3002"])] + ([("single-basin", single["3002"])] if "3002" in single else [])
        for nm, d in lines:
            m = _metrics(d.loc[a:b, "obs"].values, d.loc[a:b, "sim"].values)
            print(f"  {nm:13s} NSE {m['NSE']:.3f}  KGE {m['KGE']:.3f}  r {m['r']:.3f}  (n={m['n']})")
        print("  earlier study, setup C (CAMELS-IND inputs, 4 training years): NSE -0.193  KGE 0.380")
        fig, ax = plt.subplots(figsize=(9, 3.4))
        d = ens["3002"].loc[a:b]
        ax.plot(d.index, d.obs, color="#0b0b0b", lw=1.2, label="Observed (quality-controlled)")
        if "3002" in single:
            ax.plot(d.index, single["3002"].loc[a:b, "sim"], color="#eb6834", lw=1.2, label="Single-basin LSTM")
        ax.plot(d.index, d.sim, color="#2a78d6", lw=1.4, label="Multi-basin LSTM")
        ax.set_ylabel("Flow (mm/day)"); ax.grid(color="#e4e3df"); ax.spines[["top", "right"]].set_visible(False)
        ax.legend(frameon=False, loc="upper left"); ax.set_title("Basin 3002, test period", loc="left")
        fig.tight_layout(); fig.savefig(out / "fig_nh_3002_hydrograph.png", dpi=200); plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    for d, c, lab in [(m_all, "#2a78d6", f"Multi-basin, all {len(m_all)} basins"),
                      (m_sub, "#1baf7a", f"Multi-basin, {len(m_sub)} comparison basins"),
                      (s_sub, "#eb6834", f"Single-basin, same {len(s_sub)} basins")]:
        v = np.sort(d.NSE.clip(lower=-1).dropna().values)
        if len(v):
            ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post", color=c, lw=2, label=lab)
    ax.axvline(0, color="#8a8984", lw=1, ls="--")
    ax.set_xlim(-1, 1); ax.set_xlabel("Test NSE (values below -1 shown at -1)"); ax.set_ylabel("Share of basins")
    ax.grid(color="#e4e3df"); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, loc="upper left")
    ax.set_title("Test NSE across basins (further right = better)", loc="left")
    fig.tight_layout(); fig.savefig(out / "fig_nh_nse_cdf.png", dpi=200); plt.close(fig)
    print(f"\nSaved tables and figures in {out}")
    return summ

In [ ]:
%%writefile /content/nh_timing.py
"""Times NeuralHydrology LSTM training on any machine (laptop CPU, CSE server CPU, Colab GPU).

Two fixed benchmarks on real CAMELS-IND data (the quality-controlled data made by nh_multibasin.prepare):
  single : basin 3002 only, LSTM with 20 hidden units, MSE loss, 4 epochs   (the old single-basin setup)
  multi  : 10 basins incl. 3002, LSTM with 128 hidden units, NSE loss, 27 catchment attributes, 2 epochs
           (the multi-basin setup on a 10-basin subset, so a CPU can finish it in well under an hour)
Training 1981-2004, no validation (only training speed is measured). The first epoch includes start-up
costs, so the reported speed is the mean of the later epochs.

Usage:
  python nh_timing.py make-bench <nh_work folder> <bench folder>          (once, where the full data is)
  python nh_timing.py run <bench folder> <results folder> --device cpu --label laptop
  taskset -c 0,2,4,6,8,10,12,14 python nh_timing.py run ... --device cpu --threads 8 --label server_P8
  python nh_timing.py run <bench folder> <results folder> --device cuda:0 --label colab_t4
Output: <results folder>/nh_timing.csv (one line per benchmark and machine)
"""
import argparse, csv, json, os, platform, re, shutil, socket, subprocess, sys, time
from datetime import datetime
from pathlib import Path

import pandas as pd

TARGET = "streamflow_mm_day"
TRAIN = ("01/01/1981", "31/12/2004")


def cpu_name():
    try:
        return [l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")][0]
    except Exception:
        return platform.processor() or "cpu"


def make_bench(work_dir, bench_dir, n=10):
    work, bench = Path(work_dir), Path(bench_dir)
    info = json.loads((work / "prep_info.json").read_text())
    src = Path(info["data_dir"])
    others = sorted(b for b in info["basins"] if b != "3002")
    basins = (["3002"] if "3002" in info["basins"] else []) + others[: n - 1]
    (bench / "data" / "preprocessed").mkdir(parents=True, exist_ok=True)
    for b in basins:
        shutil.copy(src / "preprocessed" / f"{b}.csv", bench / "data" / "preprocessed" / f"{b}.csv")
    attrs = pd.read_csv(src / "attributes.csv", dtype={"gauge_id": str}).set_index("gauge_id")
    attrs.loc[basins].to_csv(bench / "data" / "attributes.csv")
    (bench / "basins_multi.txt").write_text("\n".join(basins) + "\n")
    (bench / "basins_single.txt").write_text(basins[0] + "\n")
    json.dump(dict(basins=basins, dynamic_inputs=info["dynamic_inputs"], static_attributes=info["static_attributes"]),
              open(bench / "bench_info.json", "w"), indent=1)
    size = sum(f.stat().st_size for f in (bench / "data").rglob("*") if f.is_file()) / 1e6
    print(f"Benchmark data: {len(basins)} basins ({', '.join(basins)}), {size:.0f} MB in {bench}")


def _config(name, run_dir, basin_file, data_dir, dyn, stat, hidden, loss, epochs, device):
    s = f"""experiment_name: {name}
run_dir: {run_dir}
train_basin_file: {basin_file}
validation_basin_file: {basin_file}
test_basin_file: {basin_file}
train_start_date: '{TRAIN[0]}'
train_end_date: '{TRAIN[1]}'
validation_start_date: '01/01/2005'
validation_end_date: '31/12/2009'
test_start_date: '01/01/2010'
test_end_date: '31/12/2020'
seed: 1
device: {device}
validate_every: 1000
model: cudalstm
head: regression
output_activation: linear
hidden_size: {hidden}
initial_forget_bias: 3
output_dropout: 0.4
optimizer: Adam
loss: {loss}
learning_rate:
  0: 0.001
batch_size: 256
epochs: {epochs}
clip_gradient_norm: 1
predict_last_n: 1
seq_length: 365
num_workers: 0
log_interval: 5
log_tensorboard: False
log_n_figures: 0
save_weights_every: {epochs}
dataset: camels_ind
data_dir: {data_dir}
dynamic_inputs:
""" + "".join(f"  - {v}\n" for v in dyn) + f"target_variables:\n  - {TARGET}\n"
    if stat:
        s += "static_attributes:\n" + "".join(f"  - {v}\n" for v in stat)
    return s


def _parse_log(log):
    ts = lambda line: datetime.strptime(line[:23], "%Y-%m-%d %H:%M:%S,%f")
    lines = Path(log).read_text().splitlines()
    t_load = next(ts(l) for l in lines if "Loading basin data" in l)
    t_ready = next(ts(l) for l in lines if "Create lookup table" in l)
    ep = [ts(l) for l in lines if re.search(r"Epoch \d+ average loss", l)]
    edges = [t_ready] + ep
    epochs = [(b - a).total_seconds() for a, b in zip(edges[:-1], edges[1:])]
    return (t_ready - t_load).total_seconds(), epochs


def _n_samples(data_dir, basins):
    a, b = (pd.to_datetime(x, dayfirst=True) for x in TRAIN)
    n = 0
    for k in basins:
        q = pd.read_csv(Path(data_dir) / "preprocessed" / f"{k}.csv", index_col="date", parse_dates=["date"])[TARGET]
        n += int(q.loc[a:b].notna().sum())
    return n


def run(bench_dir, out_dir, device, label, threads, which):
    bench, out = Path(bench_dir).resolve(), Path(out_dir).resolve()
    info = json.load(open(bench / "bench_info.json"))
    data = bench / "data"
    cfg_dir, run_dir = out / f"configs_{label}", out / f"runs_{label}"
    cfg_dir.mkdir(parents=True, exist_ok=True); run_dir.mkdir(parents=True, exist_ok=True)
    tests = {
        "single": dict(basins="basins_single.txt", hidden=20, loss="MSE", epochs=4, stat=[]),
        "multi": dict(basins="basins_multi.txt", hidden=128, loss="NSE", epochs=2, stat=info["static_attributes"]),
    }
    env = dict(os.environ)
    if threads:
        env.update(OMP_NUM_THREADS=str(threads), MKL_NUM_THREADS=str(threads))
    import torch
    print(f"{label}: {cpu_name()} | torch {torch.__version__} | device {device} | "
          f"threads {threads or torch.get_num_threads()}")
    path = out / "nh_timing.csv"
    for name in which:
        t = tests[name]
        basins = (bench / t["basins"]).read_text().split()
        cfg = cfg_dir / f"{name}.yml"
        cfg.write_text(_config(f"{label}_{name}", run_dir, bench / t["basins"], data, info["dynamic_inputs"],
                               t["stat"], t["hidden"], t["loss"], t["epochs"], device))
        print(f"\n== {name}: {len(basins)} basin(s), hidden {t['hidden']}, {t['epochs']} epochs ...", flush=True)
        t0 = time.perf_counter()
        r = subprocess.run(["nh-run", "train", "--config-file", str(cfg)], env=env,
                           stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
        wall = time.perf_counter() - t0
        runs = sorted(run_dir.glob(f"{label}_{name}_*"))
        if r.returncode != 0 or not runs:
            print(r.stderr[-3000:]); sys.exit(f"Training failed for {name}")
        load_s, epochs = _parse_log(runs[-1] / "output.log")
        later = epochs[1:] if len(epochs) > 1 else epochs
        mean_ep = sum(later) / len(later)
        ns = _n_samples(data, basins)
        row = dict(label=label, host=socket.gethostname(), cpu=cpu_name(), device=device,
                   threads=threads or torch.get_num_threads(), test=name, basins=len(basins), hidden=t["hidden"],
                   train_samples=ns, data_load_s=round(load_s, 1), epoch1_s=round(epochs[0], 1),
                   mean_epoch_s=round(mean_ep, 1), samples_per_s=round(ns / mean_ep, 1),
                   epochs=len(epochs), total_wall_s=round(wall, 1))
        new = not path.exists()
        with open(path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row)); new and w.writeheader(); w.writerow(row)
        print(f"   {name}: {mean_ep:.1f} s per epoch ({ns / mean_ep:.0f} samples/s), first epoch {epochs[0]:.1f} s, "
              f"data loading {load_s:.1f} s, total {wall / 60:.1f} min")
    print(f"\nSaved to {path}")


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    sub = p.add_subparsers(dest="cmd", required=True)
    m = sub.add_parser("make-bench"); m.add_argument("work_dir"); m.add_argument("bench_dir")
    m.add_argument("--n", type=int, default=10)
    r = sub.add_parser("run"); r.add_argument("bench_dir"); r.add_argument("out_dir")
    r.add_argument("--device", default="cpu"); r.add_argument("--label", default=socket.gethostname())
    r.add_argument("--threads", type=int, default=0); r.add_argument("--tests", nargs="+", default=["single", "multi"])
    a = p.parse_args()
    if a.cmd == "make-bench":
        make_bench(a.work_dir, a.bench_dir, a.n)
    else:
        run(a.bench_dir, a.out_dir, a.device, a.label, a.threads, a.tests)

In [ ]:
%%writefile /content/pangu_timing.py
"""Times Pangu-Weather (6-hour model) on the GPU or the CPU of any machine.

Only the speed is measured: the network gets a random global input of the real size
(5 variables x 13 levels x 721 x 1440 plus 4 surface fields), and each output is fed back in,
like a real forecast. The amount of computation does not depend on the input values.

Usage:
  python pangu_timing.py --device gpu --label colab_t4                 (Colab, T4 GPU)
  python pangu_timing.py --device cpu --label laptop --steps 3         (laptop / server CPU)
  taskset -c 0-15 python pangu_timing.py --device cpu --threads 8 --label server_P8 --steps 3
Output: one line per run appended to pangu_timing.csv (in --out).
"""
import argparse, csv, os, platform, resource, socket, sys, time, urllib.request
import numpy as np

URL = "https://get.ecmwf.int/repository/test-data/ai-models/pangu-weather/pangu_weather_6.onnx"


def cpu_name():
    try:
        return [l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")][0]
    except Exception:
        return platform.processor() or "cpu"


def download(path):
    if os.path.exists(path) and os.path.getsize(path) > 100e6:
        return
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    print(f"Downloading the Pangu-Weather 6-hour model (about 1.1 GB) to {path} ...")
    tmp = path + ".part"

    def hook(n, bs, total):
        if total > 0 and n % 200 == 0:
            print(f"  {100 * n * bs / total:5.1f}%", flush=True)
    urllib.request.urlretrieve(URL, tmp, hook)
    os.replace(tmp, path)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--device", choices=["cpu", "gpu"], default="cpu")
    p.add_argument("--threads", type=int, default=0, help="CPU threads for onnxruntime (0 = all)")
    p.add_argument("--steps", type=int, default=None, help="timed 6-hour steps (default: 10 GPU, 3 CPU)")
    p.add_argument("--model", default=os.path.expanduser("~/pangu_model/pangu_weather_6.onnx"))
    p.add_argument("--label", default=socket.gethostname())
    p.add_argument("--out", default=".")
    a = p.parse_args()
    steps = a.steps or (10 if a.device == "gpu" else 3)

    import onnxruntime as ort
    download(a.model)
    opts = ort.SessionOptions()
    opts.enable_cpu_mem_arena = False
    opts.enable_mem_pattern = False
    opts.enable_mem_reuse = False
    if a.threads:
        opts.intra_op_num_threads = a.threads
    providers = ([("CUDAExecutionProvider", {"arena_extend_strategy": "kSameAsRequested"}), "CPUExecutionProvider"]
                 if a.device == "gpu" else ["CPUExecutionProvider"])

    t0 = time.perf_counter()
    sess = ort.InferenceSession(a.model, sess_options=opts, providers=providers)
    load_s = time.perf_counter() - t0
    used = sess.get_providers()[0]
    if a.device == "gpu" and used != "CUDAExecutionProvider":
        sys.exit(f"GPU requested but onnxruntime is using {used}. Install onnxruntime-gpu.")
    names = [i.name for i in sess.get_inputs()]
    shapes = [i.shape for i in sess.get_inputs()]
    print(f"{a.label}: {cpu_name()} | onnxruntime {ort.__version__} | {used} | inputs {dict(zip(names, shapes))}")

    rng = np.random.default_rng(0)
    up = rng.standard_normal((5, 13, 721, 1440), dtype=np.float32)
    sf = rng.standard_normal((4, 721, 1440), dtype=np.float32)
    feed = lambda u, s: {"input": u, "input_surface": s}

    t0 = time.perf_counter(); up, sf = sess.run(None, feed(up, sf)); first_s = time.perf_counter() - t0
    print(f"  first step (includes warm-up): {first_s:.1f} s")
    times = []
    for i in range(steps):
        t0 = time.perf_counter()
        up, sf = sess.run(None, feed(up, sf))
        times.append(time.perf_counter() - t0)
        print(f"  step {i + 1}/{steps}: {times[-1]:.2f} s", flush=True)
    mean = float(np.mean(times))
    peak_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6   # ru_maxrss is in kB on Linux
    row = dict(label=a.label, host=socket.gethostname(), cpu=cpu_name(), device=a.device, provider=used,
               threads=a.threads or "all", model_load_s=round(load_s, 1), first_step_s=round(first_s, 2),
               mean_step_s=round(mean, 2), steps_timed=steps, forecast_72h_s=round(12 * mean, 1),
               peak_ram_gb=round(peak_gb, 1))
    os.makedirs(a.out, exist_ok=True)
    path = os.path.join(a.out, "pangu_timing.csv")
    new = not os.path.exists(path)
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row)); new and w.writeheader(); w.writerow(row)
    print(f"\n{a.label}: {mean:.2f} s per 6-hour step -> 72-hour forecast (12 steps) about {12 * mean:.0f} s; "
          f"peak RAM {peak_gb:.1f} GB. Saved to {path}")


if __name__ == "__main__":
    main()

In [ ]:
# 2. CAMELS-IND data (same cleaning as before) and the 10-basin benchmark set (~3-5 min)
WORK = '/content/nh_work'
os.makedirs(f'{WORK}/raw', exist_ok=True)
if not os.path.exists(f'{WORK}/raw/camels_ind.zip'):
    !wget -q -O {WORK}/raw/camels_ind.zip "https://zenodo.org/records/14999580/files/CAMELS_IND_Catchments_Streamflow_Sufficient.zip?download=1"
    !cd {WORK}/raw && unzip -q -o camels_ind.zip
import sys; sys.path.insert(0, '/content')
import nh_multibasin as nm
if not os.path.exists(f'{WORK}/prep_info.json'):
    nm.prepare(f'{WORK}/raw', WORK)
!cd /content && python nh_timing.py make-bench {WORK} /content/nh_bench

In [ ]:
# 3. NeuralHydrology training speed on the T4 GPU (a few minutes)
!cd /content && python nh_timing.py run /content/nh_bench {OUT} --device cuda:0 --label colab_t4

In [ ]:
# 4. Pangu-Weather speed on the T4 GPU (downloads the 1.1 GB model; ~5 min)
!cd /content && python pangu_timing.py --device gpu --label colab_t4 --steps 10 --model /content/pangu_model/pangu_weather_6.onnx --out {OUT}

In [ ]:
# 5. Results (also saved in MyDrive/ugp_timing)
!cat {OUT}/nh_timing.csv; echo; cat {OUT}/pangu_timing.csv

Send back the output of cell 5 (or the two CSV files from `MyDrive/ugp_timing`).